Import libraries

In [1]:
import numpy as np
import xarray as xr
import cf_xarray
from xgcm import Grid, generate_grid_ds
import gsw_xarray as gsw

import neutralocean
import neutralocean.grid.rectilinear
import neutralocean.ntp

from xgcm import Grid
import gsw_xarray as gsw

import matplotlib.pyplot as plt

import jax
from jax import numpy as jnp
import optax
import matplotlib.pyplot as plt

## At first, we process dataset to add, SA, CT, etc

Open WOA23 dataset

In [2]:
ds = xr.open_mfdataset(
    "../data/woa23*.nc", decode_times=False
).squeeze("time", drop=True)[["t_an","s_an"]]
# We only keep the objectively analyzed mean fields
ds = ds.assign_coords({"lon":ds.lon%360}).sortby("lon")
ds.lon.attrs["standard_name"] = "longitude"

/tmp/ipykernel_1719105/2558949415.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset(
/tmp/ipykernel_1719105/2558949415.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.open_mfdataset(
/tmp/ipykernel_1719105/2558949415.py:1: FutureWarning: In a future version of xarray the default value for compat wi

In [3]:
ds

<xarray.Dataset> Size: 53MB
Dimensions:  (depth: 102, lat: 180, lon: 360)
Coordinates:
  * lat      (lat) float32 720B -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * depth    (depth) float32 408B 0.0 5.0 10.0 15.0 ... 5.3e+03 5.4e+03 5.5e+03
  * lon      (lon) float32 1kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
Data variables:
    t_an     (depth, lat, lon) float32 26MB dask.array<chunksize=(51, 90, 180), meta=np.ndarray>
    s_an     (depth, lat, lon) float32 26MB dask.array<chunksize=(51, 90, 180), meta=np.ndarray>
Attributes: (12/45)
    Conventions:                     CF-1.6
    title:                           World Ocean Atlas 2023 : sea_water_pract...
    summary:                         Climatological mean salinity for the glo...
    references:                      Reagan, J.R., D. Seidov, Z. Wang, D. Duk...
    institution:                     NOAA National Centers for Environmental ...
    comment:                         Global Climatology as part of the World ...
    ...                              ...
    ncei_template_version:           NCEI_NetCDF_Grid_Template_v1.0
    license:                         These data are openly available to the p...
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    metadata_link:                   https://www.ncei.noaa.gov/products/world...
    date_created:                    2024-01-28 
    date_modified:                   2024-01-28

In [4]:
ds["p"] = gsw.p_from_z(z=-ds["depth"], lat=ds.lat)

In [5]:
# add CT and SA
ds["SA"] = ds.gsw.SA_from_SP()

In [6]:
ds["CT"] = ds.gsw.CT_from_t()

### Create xgcm.Grid

For this, we need to create some new dimensions

1. Create new dims: lon/lat/depth to x_c, y_c, z_c
1. Create new x_f, y_f, z_f dimensions that correspond to cell faces and add proper attributes
2. Add metrics
3. Create final grid

In [7]:
ds = generate_grid_ds(
    ds,
    {"X":ds.cf["longitude"].name, "Y":ds.cf["latitude"].name, "Z":ds.cf["depth"].name},
    position={'X':('center','right'),'Y':('center','right'),'Z':('center','right')}
)

In [8]:
for xyz, lld in zip(["x","y","z"],["longitude","latitude","depth"]):
    ds[f"{xyz}_c"] = (ds.cf[lld].name, np.arange(ds.cf[lld].shape[0]))
    ds = ds.swap_dims({ds.cf[lld].name:f"{xyz}_c"})
    ds[f"{xyz}_f"] = (ds.cf[lld].name+"_right", np.arange(ds.cf[lld].shape[0]))
    ds = ds.swap_dims({ds.cf[lld].name+"_right":f"{xyz}_f"})
    ds[f"{xyz}_c"].attrs["axis"] = ds[ds.cf[lld].name].attrs.pop("axis")
    ds[f"{xyz}_f"].attrs["axis"] = ds[ds.cf[lld].name+"_right"].attrs.pop("axis")
    ds[f"{xyz}_f"].attrs["c_grid_axis_shift"] = ds[ds.cf[lld].name+"_right"].attrs.pop("c_grid_axis_shift")
    ds[ds.cf[lld].name+"_right"].attrs["standard_name"] = lld

In [9]:
grid = Grid(ds, periodic=["X"])
grid

<xgcm.Grid>
Z Axis (not periodic, boundary=None):
  * center   z_c --> right
  * right    z_f --> center
X Axis (periodic, boundary=None):
  * center   x_c --> right
  * right    x_f --> center
Y Axis (not periodic, boundary=None):
  * center   y_c --> right
  * right    y_f --> center

In [10]:
dlong = 1
dlonc = 1

dlatg =1
dlatc = 1

In [11]:
def dll_dist(dlon, dlat, lon, lat):
        """Converts lat/lon differentials into distances in meters

        PARAMETERS
        ----------
        dlon : xarray.DataArray longitude differentials
        dlat : xarray.DataArray latitude differentials
        lon  : xarray.DataArray longitude values
        lat  : xarray.DataArray latitude values

        RETURNS
        -------
        dx  : xarray.DataArray distance inferred from dlon
        dy  : xarray.DataArray distance inferred from dlat
        """

        distance_1deg_equator = 111000.0
        dx = dlon * np.cos(np.deg2rad(lat)) * distance_1deg_equator
        dy = ((lon * 0) + 1) * dlat * distance_1deg_equator
        return dx, dy

ds.coords['e1tu'], e2 = dll_dist(dlonc, dlatc, ds.lon, ds.lat)
# e1tu is at u and t points => function of y_c
ds.coords['e2'] = np.nanmean(e2)
ds.coords['e1vf'], e2 = dll_dist(dlonc, dlatc, ds.lon, ds.lat_right)
# e1vf is at v and f points => function of y_f

In [12]:
ds.coords["e3w"] = grid.diff(
    ds["depth"],
    "Z",
    boundary="fill",
    fill_value=2*ds["depth"].values[-1] - ds["depth"].values[-2]
)

/home/romain/Documents/work/2025-Jaxathon/neutral-density/.venv/lib/python3.12/site-packages/xgcm/grid_ufunc.py:832: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  out_dim: grid._ds.dims[out_dim] for arg in out_core_dims for out_dim in arg


In [13]:
ds.coords["e3t"] = grid.diff(
    ds["depth_right"],
    "Z",
    boundary="fill",
    fill_value=-ds["depth_right"].values[0]
)

### We recreate the xgcm.Grid with the new dataset

In [14]:
grid = Grid(ds, periodic=["X"], metrics={"X":["e1tu","e1vf"], "Y":["e2"], "Z":["e3t","e3w"]})
grid

<xgcm.Grid>
Z Axis (not periodic, boundary=None):
  * center   z_c --> right
  * right    z_f --> center
X Axis (periodic, boundary=None):
  * center   x_c --> right
  * right    x_f --> center
Y Axis (not periodic, boundary=None):
  * center   y_c --> right
  * right    y_f --> center

In [15]:
# we transpose the dataset to get (x,y,z) order
ds = ds.transpose(
    "x_c", "x_f", "y_c", "y_f", "z_c", "z_f", ...
)

In [22]:
# we transpose the dataset to get (x,y,z) order
ds = ds.transpose(
    "x_c", "x_f", "y_c", "y_f", "z_c", "z_f", ...
)

In [23]:
# save for later use
ds.to_netcdf("dataset.nc")
ds.close()